# Real ANTs SyN Warp — Analysis & Correction Pipeline

Loads the ANTs SyN displacement field from registering
`B0039_brain_25.nii.gz` (moving) to `average_template_25.nii.gz` (fixed).

**Key observation:** ANTs SyN is a *diffeomorphic* registration method —
it is designed by construction to produce deformation fields with strictly
positive Jacobian determinants.  This notebook therefore serves two roles:

1. **Verification** — confirm the warp is fold-free and visualise the
   Jdet distribution and planned warping.
2. **Correction pipeline** — the same infrastructure applies directly
   to warps from non-diffeomorphic methods (Elastix B-spline, optical
   flow, etc.) that *do* produce folding.

**Convention:** ANTs warps are pull-back (backward) maps — for every
voxel in **fixed** space the field gives the displacement to the
corresponding location in **moving** space.  This matches dvfopt's
`[dz, dy, dx]` convention exactly.

**Two parts:**

* **Part A (always runs):** analysis on a small axial subset —
  by default the 10 slices with the largest deformation magnitude.
* **Part B (opt-in):** same pipeline on the full 3-D volume.
  Set `RUN_FULL_VOLUME = True` to enable.


## Imports

In [ ]:
import time

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from dvfopt import jacobian_det2D, iterative_serial
from dvfopt.viz import plot_grid_before_after


## Configuration

In [ ]:
WARP_PATH   = "../../data/test_cases/ants_b0039/syn_1Warp.nii.gz"
FIXED_PATH  = "../../data/mouse_brain/average_template_25.nii.gz"
MOVING_PATH = "../../data/mouse_brain/B0039_brain_25.nii.gz"

JDET_THRESHOLD = 0.01

# Part A: axial slice indices to analyse.
# None = auto-select N_SUBSET slices with lowest min Jdet.
# Set explicitly to override, e.g. list(range(150, 160)).
SUBSET_SLICES = None
N_SUBSET = 10

# Part B: process the full volume (slow)
RUN_FULL_VOLUME = False


## Load warp field and scan all slices

In [ ]:
print("Loading warp field...")
nii = nib.load(WARP_PATH)
warp_data = nii.get_fdata()          # (X, Y, Z, 1, 3) ITK xyz
print(f"  Raw shape : {warp_data.shape}")
print(f"  Voxel size: {nii.header.get_zooms()[:3]}")

if warp_data.ndim == 5 and warp_data.shape[3] == 1:
    warp_data = warp_data[:, :, :, 0, :]

# ITK xyz -> dvfopt [dz, dy, dx] — shape (3, NX, NY, NZ)
warp_zyx = np.stack([
    warp_data[..., 2],   # dz
    warp_data[..., 1],   # dy
    warp_data[..., 0],   # dx
], axis=0)
NX, NY, NZ = warp_zyx.shape[1], warp_zyx.shape[2], warp_zyx.shape[3]
print(f"  dvfopt shape: {warp_zyx.shape}  [dz, dy, dx]  (NX={NX}, NY={NY}, NZ={NZ})")

# ------------------------------------------------------------------
# Full scan: Jdet and displacement magnitude per axial slice
# ------------------------------------------------------------------
print(f"\nScanning all {NZ} axial slices...")
scan_stats = {}   # z -> dict(n_neg, min_jdet, max_jdet, max_disp)

for z in range(NZ):
    dy = warp_zyx[1, :, :, z]
    dx = warp_zyx[2, :, :, z]
    phi = np.stack([dy, dx])
    jac = jacobian_det2D(phi)
    jmin = float(jac.min())
    jmax = float(jac.max())
    n_neg = int((jac <= 0).sum())
    max_disp = float(np.sqrt(dy**2 + dx**2).max())
    scan_stats[z] = dict(n_neg=n_neg, min_jdet=jmin, max_jdet=jmax, max_disp=max_disp)

n_with_neg = sum(1 for s in scan_stats.values() if s['n_neg'] > 0)
all_jmins = [s['min_jdet'] for s in scan_stats.values()]
print(f"  Slices with neg Jdet (< 0): {n_with_neg} / {NZ}")
print(f"  Global min Jdet: {min(all_jmins):+.6f}")
print(f"  Global max Jdet: {max(s['max_jdet'] for s in scan_stats.values()):+.6f}")
print(f"  Global max disp: {max(s['max_disp'] for s in scan_stats.values()):.4f}")

if n_with_neg == 0:
    print("\n  NOTE: ANTs SyN produced a fold-free (diffeomorphic) warp.")
    print("  The correction pipeline below runs but finds nothing to fix.")
    print("  Apply to warps from non-diffeomorphic methods to see corrections.")

# Auto-select subset: slices with lowest min_jdet (most stressed)
slices_by_jmin = sorted(scan_stats.keys(), key=lambda z: scan_stats[z]['min_jdet'])
_auto_subset = sorted(slices_by_jmin[:N_SUBSET])

if SUBSET_SLICES is None:
    SUBSET_SLICES = _auto_subset
    print(f"\nAuto-selected subset (lowest min_jdet): {SUBSET_SLICES}")
else:
    print(f"\nUser-specified subset: {SUBSET_SLICES}")


## Load fixed image

In [ ]:
print("Loading fixed image for overlay...")
fixed_nii = nib.load(FIXED_PATH)
fixed_vol = fixed_nii.get_fdata().astype(np.float32)
fixed_vol = (fixed_vol - fixed_vol.min()) / (fixed_vol.max() - fixed_vol.min() + 1e-8)
print(f"  Shape: {fixed_vol.shape}")


## Helpers

In [ ]:
def extract_slice_deformation(z):
    deformation = np.zeros((3, 1, NX, NY), dtype=np.float64)
    deformation[1, 0] = warp_zyx[1, :, :, z]
    deformation[2, 0] = warp_zyx[2, :, :, z]
    return deformation


def show_jdet_map(jac2d, title, ax, fixed_slice=None):
    """Display a (NX, NY) Jdet map, with optional fixed-image underlay."""
    jac2d = np.squeeze(jac2d)           # (NX, NY)
    if fixed_slice is not None:
        # fixed_slice shape (NX, NY); imshow: rows=NX (y-axis), cols=NY (x-axis)
        ax.imshow(fixed_slice, cmap="gray", alpha=0.4, vmin=0, vmax=1, origin="upper")
    vmax = max(abs(float(jac2d.min())), abs(float(jac2d.max())), 1.0)
    im = ax.imshow(jac2d, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   alpha=0.7 if fixed_slice is not None else 1.0, origin="upper")
    n_neg = int((jac2d <= 0).sum())
    ax.set_title(title + chr(10) + f"neg={n_neg}  min={jac2d.min():+.4f}", fontsize=8)
    ax.axis("off")
    return im


def correct_slice(z, verbose=0):
    deformation = extract_slice_deformation(z)
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg = int((jac_init <= 0).sum())

    if n_neg == 0:
        return phi_init, phi_init, jac_init, jac_init, 0.0, 0.0

    t0 = time.perf_counter()
    phi = iterative_serial(deformation.copy(), verbose=verbose, threshold=JDET_THRESHOLD)
    elapsed = time.perf_counter() - t0
    jac_final = jacobian_det2D(phi)
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))
    return phi_init, phi, jac_init, jac_final, elapsed, l2


---
## Part A — Subset analysis

### Jdet distribution (full volume)

In [ ]:
# Jdet distribution across the full volume (sampled every 5th slice)
all_jdets = []
for z in range(0, NZ, 5):
    dy = warp_zyx[1, :, :, z]
    dx = warp_zyx[2, :, :, z]
    jac = jacobian_det2D(np.stack([dy, dx]))
    all_jdets.append(np.squeeze(jac).ravel())

all_jdets = np.concatenate(all_jdets)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)

ax = axes[0]
ax.hist(all_jdets, bins=80, color="tab:blue", alpha=0.8)
ax.axvline(0, color="red", lw=1.5, linestyle="--", label="Jdet=0 (fold)")
ax.axvline(JDET_THRESHOLD, color="orange", lw=1.5, linestyle="--",
           label=f"threshold={JDET_THRESHOLD}")
ax.set_xlabel("Jacobian determinant")
ax.set_ylabel("Count")
ax.set_title("Jdet distribution (full volume, sampled)")
ax.legend(fontsize=8)

ax = axes[1]
min_jdets = [scan_stats[z]['min_jdet'] for z in range(NZ)]
ax.plot(range(NZ), min_jdets, lw=0.8, color="tab:blue")
ax.axhline(0, color="red", lw=1, linestyle="--", label="Jdet=0")
ax.axhline(JDET_THRESHOLD, color="orange", lw=1, linestyle="--",
           label=f"threshold={JDET_THRESHOLD}")
ax.set_xlabel("Axial slice index (z)")
ax.set_ylabel("Min Jdet")
ax.set_title("Min Jdet per axial slice")
ax.legend(fontsize=8)

plt.suptitle("Jacobian determinant — real ANTs SyN warp (B0039 -> avg template)", fontsize=11)
plt.show()
print(f"Global min Jdet: {float(np.array(min_jdets).min()):+.6f}")
print(f"Fraction of pixels with Jdet < 1.0: {(all_jdets < 1.0).mean()*100:.1f}%")


### Survey subset slices

In [ ]:
# Survey the subset slices
print(f"Subset: {SUBSET_SLICES}")
print(f"{'z':>6s}  {'neg_jdet':>8s}  {'min_jdet':>10s}  {'max_jdet':>10s}  {'max_disp':>10s}")
print("-" * 50)
for z in SUBSET_SLICES:
    s = scan_stats[z]
    print(f"{z:>6d}  {s['n_neg']:>8d}  {s['min_jdet']:>10.5f}  {s['max_jdet']:>10.5f}  {s['max_disp']:>10.4f}")


### Planned warping

In [ ]:
# Planned warping: deformed grid overlaid on fixed image
# Orientation: rows=NX (y-axis), cols=NY (x-axis) — no transposition
stride = 8

n = len(SUBSET_SLICES)
ncols = min(5, n)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows),
                         constrained_layout=True)
axes_flat = np.array(axes).flat

for ax, z in zip(axes_flat, SUBSET_SLICES):
    dy = warp_zyx[1, :, :, z]   # shape (NX, NY)
    dx = warp_zyx[2, :, :, z]

    # Deformed grid: row-index (axis 0) is y, col-index (axis 1) is x
    rows = np.arange(0, NX, stride)
    cols = np.arange(0, NY, stride)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")
    def_y = rr + dy[rr, cc]   # deformed row coord (y-axis in plot)
    def_x = cc + dx[rr, cc]   # deformed col coord (x-axis in plot)

    # Fixed image underlay: shape (NX, NY) — no .T needed
    if z < fixed_vol.shape[2]:
        ax.imshow(fixed_vol[:, :, z], cmap="gray", vmin=0, vmax=1, origin="upper")

    for i in range(def_y.shape[0]):
        ax.plot(def_x[i, :], def_y[i, :], "r-", lw=0.5, alpha=0.7)
    for j in range(def_y.shape[1]):
        ax.plot(def_x[:, j], def_y[:, j], "r-", lw=0.5, alpha=0.7)

    ax.set_xlim(0, NY)
    ax.set_ylim(NX, 0)
    s = scan_stats[z]
    ax.set_title(f"z={z}" + chr(10) + f"min_jdet={s['min_jdet']:+.4f}", fontsize=8)
    ax.axis("off")

for ax in list(axes_flat)[len(SUBSET_SLICES):]:
    ax.axis("off")

plt.suptitle("Planned warping — subset slices  (pull-back displacement field)", fontsize=11)
plt.show()


### Jdet maps

In [ ]:
# Jdet maps for the subset (overlaid on fixed image)
n = len(SUBSET_SLICES)
ncols = min(5, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows),
                         constrained_layout=True)
axes_flat = np.array(axes).flat
ims = []

for ax, z in zip(axes_flat, SUBSET_SLICES):
    dy = warp_zyx[1, :, :, z]
    dx = warp_zyx[2, :, :, z]
    jac = np.squeeze(jacobian_det2D(np.stack([dy, dx])))
    fixed_sl = fixed_vol[:, :, z] if z < fixed_vol.shape[2] else None
    im = show_jdet_map(jac, f"z={z}", ax, fixed_sl)
    ims.append(im)

for ax in list(axes_flat)[n:]:
    ax.axis("off")

plt.colorbar(ims[0], ax=list(np.array(axes).flat)[:n], shrink=0.6, label="Jacobian det")
plt.suptitle("Jacobian determinant — subset slices", fontsize=11)
plt.show()


### Correction pipeline

In [ ]:
# Run correction pipeline on subset
# (for a fold-free SyN warp this is a no-op, but the pipeline is validated)
subset_results = {}

for z in SUBSET_SLICES:
    phi_init, phi, jac_init, jac_final, elapsed, l2 = correct_slice(z, verbose=0)
    n_before = int((jac_init <= 0).sum())
    n_after  = int((jac_final <= 0).sum())
    subset_results[z] = dict(
        phi_init=phi_init, phi=phi,
        jac_init=jac_init, jac_final=jac_final,
        n_before=n_before, n_after=n_after,
        elapsed=elapsed, l2=l2,
    )
    status = "no folding" if n_before == 0 else ("OK" if n_after == 0 else f"PARTIAL ({n_after} remain)")
    print(f"  z={z:4d}  neg: {n_before:4d} -> {n_after:4d}  "
          f"min: {float(jac_init.min()):+.5f} -> {float(jac_final.min()):+.5f}  "
          f"L2={l2:.4f}  t={elapsed:.2f}s  [{status}]")


### Grid before/after

In [ ]:
# Grid before/after for slices that had folding
folded = [z for z in SUBSET_SLICES if subset_results[z]['n_before'] > 0]
if folded:
    for z in folded:
        deformation = extract_slice_deformation(z)
        plot_grid_before_after(deformation, subset_results[z]['phi'],
                               title=f"z={z} — real ANTs SyN warp")
else:
    print("No folded slices in subset — grid comparison not applicable.")
    print("(ANTs SyN is diffeomorphic by design.)")


### Summary table

In [ ]:
print(f"{'z':>6s}  {'neg_before':>10s}  {'neg_after':>10s}  {'min_before':>12s}  {'min_after':>12s}  {'L2':>8s}  {'Time(s)':>8s}")
print("-" * 70)
for z in SUBSET_SLICES:
    r = subset_results[z]
    print(f"{z:>6d}  {r['n_before']:>10d}  {r['n_after']:>10d}  "
          f"{float(r['jac_init'].min()):>12.5f}  {float(r['jac_final'].min()):>12.5f}  "
          f"{r['l2']:>8.4f}  {r['elapsed']:>8.2f}")


---
## Part B — Full volume correction

Set `RUN_FULL_VOLUME = True` in Configuration to enable.
Processes all axial slices that contain negative Jacobians.


In [ ]:
if not RUN_FULL_VOLUME:
    print("Part B disabled (set RUN_FULL_VOLUME = True to enable).")
else:
    print("Part B: full volume correction")
    slices_to_correct = sorted(z for z, s in scan_stats.items() if s['n_neg'] > 0)
    print(f"  Slices with neg Jdet: {len(slices_to_correct)} / {NZ}")

    if not slices_to_correct:
        print("  No slices with negative Jacobians found — warp is fold-free.")
    else:
        vol_results = {}
        t_total = 0.0
        for i, z in enumerate(slices_to_correct):
            phi_init, phi, jac_init, jac_final, elapsed, l2 = correct_slice(z, verbose=0)
            n_before = int((jac_init <= 0).sum())
            n_after  = int((jac_final <= 0).sum())
            t_total += elapsed
            vol_results[z] = dict(n_before=n_before, n_after=n_after, elapsed=elapsed, l2=l2,
                                  min_before=float(jac_init.min()), min_after=float(jac_final.min()))
            if (i + 1) % 10 == 0 or n_after > 0:
                print(f"  [{i+1:4d}/{len(slices_to_correct)}] z={z:4d}  "
                      f"neg: {n_before}->{n_after}  t={elapsed:.2f}s")

        n_fully = sum(1 for r in vol_results.values() if r['n_after'] == 0)
        print(f"\nDone in {t_total:.1f}s  |  fully corrected: {n_fully}/{len(slices_to_correct)}")

        slices_sorted = sorted(vol_results.keys())
        before = [vol_results[z]['n_before'] for z in slices_sorted]
        after  = [vol_results[z]['n_after']  for z in slices_sorted]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
        x = np.arange(len(slices_sorted))
        axes[0].bar(x, before, 0.6, label="Before", color="tab:red",   alpha=0.7)
        axes[0].bar(x, after,  0.6, label="After",  color="tab:green", alpha=0.7)
        axes[0].set_ylabel("Neg Jdet pixels")
        axes[0].set_xlabel("Axial slice index")
        axes[0].set_title("Neg Jdet per slice")
        axes[0].legend()

        axes[1].bar(x, [vol_results[z]['elapsed'] for z in slices_sorted], 0.6,
                    color="tab:blue", alpha=0.7)
        axes[1].set_ylabel("Correction time (s)")
        axes[1].set_xlabel("Axial slice index")
        axes[1].set_title("Correction time per slice")

        plt.suptitle("Full volume correction — real ANTs SyN warp", fontsize=12)
        plt.show()
